[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/spatialft/spatialft.github.io/blob/main/notebooks/04_eval_comparison.ipynb)

# Notebook 4 — Evaluation Comparison

Run the fine-tuned model on the eval set and compare baseline vs fine-tuned accuracy.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/spatialft.github.io')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/spatialft/spatialft.github.io.git', str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.colab_utils import bootstrap_colab_repo, get_repo_paths, publish_artifacts, require_local_adapter

REPO = bootstrap_colab_repo(REPO)
PATHS = get_repo_paths(REPO)
print(f'Repo ready at {REPO}')
print(f'Using local repo storage: {PATHS["repo_root"]}')


In [ ]:
import json
from pathlib import Path
from tqdm import tqdm
import torch
import matplotlib.pyplot as plt

from src.dataset import load_stepgame, format_prompt
from src.eval import evaluate, save_results, load_predictions, extract_answer


In [ ]:
ADAPTER_PATH = require_local_adapter(REPO)
EVAL_PATH = PATHS['data_root'] / 'eval' / 'stepgame_eval.json'
OUT_PATH = PATHS['results_root'] / 'finetuned' / 'predictions.json'
SCORES_PATH = PATHS['results_root'] / 'finetuned' / 'scores.json'
EXAMPLES_OUT = PATHS['results_root'] / 'examples.json'
BASELINE_PREDS = PATHS['results_root'] / 'baseline' / 'predictions.json'
MAX_SEQ_LENGTH = 512
MAX_NEW_TOKENS = 64  # Direct-answer prompting keeps generations short for the 350M model
BATCH_SIZE = 8

print(f'Loading adapter from: {ADAPTER_PATH}')
print(f'Using baseline predictions from: {BASELINE_PREDS}')


In [ ]:
USE_UNSLOTH = False

try:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=ADAPTER_PATH,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    USE_UNSLOTH = True
    print("Unsloth loaded successfully")
except Exception as e:
    print(f"Unsloth unavailable ({e}), falling back to transformers+peft")
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import PeftModel

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    base_model_id = "LiquidAI/LFM2-350M"
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
    tokenizer.pad_token = tokenizer.eos_token
    base = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model = PeftModel.from_pretrained(base, ADAPTER_PATH)
    model.eval()


In [ ]:
examples_raw = load_stepgame(EVAL_PATH)
predictions = []

for i in tqdm(range(0, len(examples_raw), BATCH_SIZE)):
    batch = examples_raw[i : i + BATCH_SIZE]
    prompts = [format_prompt(ex['story'], ex['question']) for ex in batch]

    inputs = tokenizer(
        prompts,
        return_tensors='pt',
        padding=True,
        truncation=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs['input_ids'].shape[1]
    for ex, output in zip(batch, outputs):
        generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)
        predictions.append({
            'story': ex['story'],
            'question': ex['question'],
            'answer': ex['answer'],
            'prediction': generated,
            'k': ex.get('k'),
        })

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_PATH, 'w') as f:
    json.dump(predictions, f, indent=2)

ft_results = evaluate(predictions)
save_results(ft_results, SCORES_PATH)


In [ ]:
with open(PATHS['results_root'] / 'baseline' / 'scores.json') as f:
    baseline = json.load(f)

print(f'Baseline accuracy:  {baseline["accuracy"]:.3f}')
print(f'Fine-tuned accuracy: {ft_results["accuracy"]:.3f}')
print(f'Delta: {ft_results["accuracy"] - baseline["accuracy"]:+.3f}')


In [ ]:
# Per-hop comparison chart
k_vals = sorted(set(
    int(k.replace('accuracy_k', ''))
    for k in baseline if k.startswith('accuracy_k')
))

base_scores = [baseline.get(f'accuracy_k{k}', 0) for k in k_vals]
ft_scores = [ft_results.get(f'accuracy_k{k}', 0) for k in k_vals]
COMPARISON_PLOT = PATHS['results_root'] / 'comparison.png'

x = range(len(k_vals))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i - width/2 for i in x], base_scores, width, label='Baseline')
ax.bar([i + width/2 for i in x], ft_scores, width, label='Fine-tuned')
ax.set_xticks(list(x))
ax.set_xticklabels([f'k={k}' for k in k_vals])
ax.set_ylabel('Accuracy')
ax.set_title('Spatial Reasoning Accuracy by Hop Level')
ax.legend()
plt.tight_layout()
plt.savefig(COMPARISON_PLOT, dpi=150)
plt.show()
print(f'Saved comparison plot to {COMPARISON_PLOT}')


In [ ]:
# Export examples where baseline was wrong and fine-tuned is right
# These feed the project landing page at spatialft.github.io
baseline_preds = load_predictions(BASELINE_PREDS)

showcase = []
seen_k = {}

for ft_pred, base_pred in zip(predictions, baseline_preds):
    gold = ft_pred['answer'].strip().lower()
    base_ans = extract_answer(base_pred['prediction']) or ''
    ft_ans = extract_answer(ft_pred['prediction']) or ''
    k = ft_pred.get('k')

    # Only keep: baseline wrong, fine-tuned right
    if base_ans != gold and ft_ans == gold:
        if seen_k.get(k, 0) < 2:
            showcase.append({
                'story': ft_pred['story'],
                'question': ft_pred['question'],
                'answer': gold,
                'baseline': base_ans,
                'finetuned': ft_ans,
                'k': k,
            })
            seen_k[k] = seen_k.get(k, 0) + 1

showcase = sorted(showcase, key=lambda x: x['k'] or 0)[:8]

with open(EXAMPLES_OUT, 'w') as f:
    json.dump(showcase, f, indent=2)

print(f'Saved {len(showcase)} showcase examples to {EXAMPLES_OUT}')
for ex in showcase:
    print(f'  k={ex["k"]} | gold={ex["answer"]} | base={ex["baseline"]} | ft={ex["finetuned"]}')


In [ ]:
publish_artifacts(
    [
        'results/finetuned/scores.json',
        'results/examples.json',
        'results/comparison.png',
    ],
    'Add finetuned scores and examples [notebook 04]',
    repo_dir=REPO,
)
